In [352]:
import csv, json
import polars as pl ## fast pandas
## pydantic_jsonschema enables class creation from external .json file:
from pydantic_jsonschema import Schema, to_model

In [353]:
## 'model name': {'path': 'path_to_json_schema'}
models = {
    'EventCore': {'path': 'static/event_core.json'},
    'Releve': {'path': 'static/releve.json'}
}

In [354]:
## use this model (schema) vor validation:
model_name='EventCore'

In [355]:
## create validation model "my_model" from external JSONschema:
with open(models[model_name]['path'], 'r') as file:
    my_model = to_model(
        Schema.model_validate(json.load(file)),
        model_name=model_name
    )

In [356]:
## read in CSV for rowwise validation,
df = polars.read_csv(
    source = 'static/data.csv',
    ## override automatic datatyping like so:
    schema_overrides = {
        "decimalLatitude": polars.datatypes.Float16,
        "decimalLongitude": polars.datatypes.Float16
    },
    ## set values which cannot be converted to desired
    ## datatype to null (have validation catch them later):
    ignore_errors = True
)

In [357]:
## validate a given df row against chosen schema:
def validate (row):
    try:
        my_model.model_validate(row)
        return "OK"
    except ValueError as e:
        errors = e.errors()
        locs = [error["loc"][0] for error in errors]
        msgs = [error["msg"] for error in errors]
        return ";".join([": ".join(b) for b in zip(locs, msgs)])
    
## validate df rowwise:
validation=[validate(row) for row in df.iter_rows(named = True)]

In [358]:
## add validation results as last column:
df.replace_column(
    df.shape[1]-1,
    pl.Series("validation", validation)
)

siteID,decimalLongitude,decimalLatitude,countryCode,validation
str,f16,f16,str,str
"""site1""",16.5,90.0,"""AT""","""OK"""
"""site2""",16.5,90.0,"""ATX""","""countryCode: Input should be '…"
